# Prepare a 20,000-image LAION-Art pool

This notebook follows the official [`img2dataset` LAION-Art recipe](https://github.com/rom1504/img2dataset/blob/main/dataset_examples/laion-art.md): cache the official Hugging Face parquet, select rows reproducibly, and let `img2dataset` perform parallel URL downloads. It does **not** create captions or concept groups.

AMP says that its images are resized to 1024×1024, but neither the paper nor repository specifies an interpolation or crop policy for constructing this pool. This notebook therefore uses a documented deterministic policy: resize directly to 1024×1024 with Pillow LANCZOS (no crop). An existing, valid 1024×1024 PNG is copied byte-for-byte instead of being re-encoded.

Install notebook-only dependencies in the active environment if needed:

```bash
uv pip install img2dataset pyarrow pandas pillow
```


In [ ]:
from concurrent.futures import ProcessPoolExecutor
import json
import os
from pathlib import Path
import random
import shutil
import subprocess
import urllib.request

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from PIL import Image, ImageOps

# Configuration
N = 20_000
RANDOM_SEED = 2025
METADATA_URL = "https://huggingface.co/datasets/laion/laion-art/resolve/main/laion-art.parquet"
METADATA_CACHE = Path("dataset/laion_art/cache/laion-art.parquet")
TEMP_DOWNLOAD_ROOT = Path("dataset/laion_art/tmp")
FINAL_IMAGE_DIR = Path("dataset/laion_art/clean")
FINAL_METADATA_CSV = Path("dataset/laion_art/metadata.csv")
CPU_WORKERS = max(1, (os.cpu_count() or 2) - 1)
DOWNLOAD_PROCESSES = min(16, CPU_WORKERS)
DOWNLOAD_THREADS = 32
TARGET_SIZE = (1024, 1024)

# Selection-specific paths prevent an incremental download from mixing seeds/subsets.
RUN_DIR = TEMP_DOWNLOAD_ROOT / f"n{N}_seed{RANDOM_SEED}"
SELECTED_PARQUET = RUN_DIR / "selected.parquet"
RAW_IMAGE_DIR = RUN_DIR / "downloads"

for directory in (METADATA_CACHE.parent, RUN_DIR, FINAL_IMAGE_DIR):
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
def download_once(url, destination, chunk_bytes=1024 * 1024):
    """Stream to an atomic temporary file; reuse a nonempty cached parquet."""
    if destination.is_file() and destination.stat().st_size:
        return destination
    temporary = destination.with_suffix(destination.suffix + ".part")
    with urllib.request.urlopen(url) as response, temporary.open("wb") as output:
        shutil.copyfileobj(response, output, length=chunk_bytes)
    temporary.replace(destination)
    return destination


def reservoir_sample_parquet(source, count, seed, batch_size=8192):
    """Select count rows deterministically in one streaming pass over parquet metadata."""
    rng = random.Random(seed)
    sample = []
    seen = 0
    parquet = pq.ParquetFile(source)
    for batch in parquet.iter_batches(batch_size=batch_size, use_threads=True):
        for row in batch.to_pylist():
            row["metadata_row"] = seen
            if seen < count:
                sample.append(row)
            else:
                replacement = rng.randrange(seen + 1)
                if replacement < count:
                    sample[replacement] = row
            seen += 1
    if seen < count:
        raise ValueError(f"Requested {count:,} rows, but metadata contains only {seen:,}.")
    sample.sort(key=lambda row: row["metadata_row"])
    for row in sample:
        row["sample_id"] = f"{row['metadata_row']:012d}"
    return sample, seen


download_once(METADATA_URL, METADATA_CACHE)
if not SELECTED_PARQUET.exists():
    selected_rows, total_rows = reservoir_sample_parquet(METADATA_CACHE, N, RANDOM_SEED)
    pq.write_table(pa.Table.from_pylist(selected_rows), SELECTED_PARQUET, compression="zstd")
else:
    total_rows = pq.ParquetFile(METADATA_CACHE).metadata.num_rows

print(f"Selected {pq.ParquetFile(SELECTED_PARQUET).metadata.num_rows:,} of {total_rows:,} rows")


In [ ]:
# Keep img2dataset's official URL/TEXT columns and retain sample_id in JSON sidecars.
# incremental mode skips shards completed by an earlier interrupted run.
URL_COLUMN = "URL"
CAPTION_COLUMN = "TEXT"
selected_columns = set(pq.read_schema(SELECTED_PARQUET).names)
if URL_COLUMN not in selected_columns:
    raise KeyError(f"Expected official URL column {URL_COLUMN!r}; found {sorted(selected_columns)}")
caption_args = ["--caption_col", CAPTION_COLUMN] if CAPTION_COLUMN in selected_columns else []

command = [
    "img2dataset",
    "--url_list", str(SELECTED_PARQUET),
    "--input_format", "parquet",
    "--url_col", URL_COLUMN,
    *caption_args,
    "--output_format", "files",
    "--output_folder", str(RAW_IMAGE_DIR),
    "--resize_mode", "no",  # Preserve downloaded bytes for controlled final processing.
    "--processes_count", str(DOWNLOAD_PROCESSES),
    "--thread_count", str(DOWNLOAD_THREADS),
    "--number_sample_per_shard", "1000",
    "--save_additional_columns", json.dumps(["sample_id"]),
    "--incremental_mode", "incremental",
]
print("Running:", " ".join(command))
subprocess.run(command, check=True)


In [ ]:
def valid_final_png(path):
    try:
        with Image.open(path) as image:
            image.verify()
        with Image.open(path) as image:
            return image.format == "PNG" and image.size == TARGET_SIZE
    except (OSError, ValueError):
        return False


def find_downloads(raw_directory):
    """Map sample_id to downloaded file using img2dataset's JSON sidecars."""
    found = {}
    for sidecar in raw_directory.rglob("*.json"):
        try:
            payload = json.loads(sidecar.read_text())
        except (OSError, json.JSONDecodeError):
            continue
        sample_id = payload.get("sample_id")
        if sample_id is None:
            continue
        candidates = [
            path for path in sidecar.parent.glob(sidecar.stem + ".*")
            if path.suffix.lower() not in {".json", ".txt"}
        ]
        if candidates:
            found[str(sample_id)] = candidates[0]
    return found


def process_image(task):
    """Validate, normalize to the required PNG/size, and delete raw only on success."""
    sample_id, raw_path_string, final_path_string = task
    raw_path = Path(raw_path_string) if raw_path_string else None
    final_path = Path(final_path_string)

    if valid_final_png(final_path):
        if raw_path and raw_path.exists() and raw_path != final_path:
            raw_path.unlink()
        return sample_id, "complete", "", str(final_path)
    if raw_path is None or not raw_path.is_file():
        return sample_id, "download_failed", "No downloaded image was recorded", ""

    temporary = final_path.with_suffix(".png.part")
    try:
        with Image.open(raw_path) as image:
            image.verify()
        with Image.open(raw_path) as image:
            original_format = image.format
            original_size = image.size
            if original_format == "PNG" and original_size == TARGET_SIZE:
                shutil.copyfile(raw_path, temporary)
            else:
                image = ImageOps.exif_transpose(image).convert("RGB")
                if image.size != TARGET_SIZE:
                    image = image.resize(TARGET_SIZE, resample=Image.Resampling.LANCZOS)
                image.save(temporary, format="PNG", optimize=False)
        if not valid_final_png(temporary):
            raise ValueError("Final PNG validation failed")
        temporary.replace(final_path)
        raw_path.unlink()
        return sample_id, "complete", "", str(final_path)
    except Exception as error:
        temporary.unlink(missing_ok=True)
        return sample_id, "decode_failed", f"{type(error).__name__}: {error}", ""


In [ ]:
# Only metadata for the selected 20,000 rows is held in memory; image pixels never are.
metadata = pq.read_table(SELECTED_PARQUET).to_pandas()
metadata["sample_id"] = metadata["sample_id"].astype(str)
downloaded = find_downloads(RAW_IMAGE_DIR)
tasks = [
    (
        sample_id,
        str(downloaded[sample_id]) if sample_id in downloaded else "",
        str(FINAL_IMAGE_DIR / f"{sample_id}.png"),
    )
    for sample_id in metadata["sample_id"]
]

with ProcessPoolExecutor(max_workers=CPU_WORKERS) as executor:
    outcomes = list(executor.map(process_image, tasks, chunksize=16))

status = pd.DataFrame(outcomes, columns=["sample_id", "status", "error", "final_path"])
metadata = metadata.merge(status, on="sample_id", how="left", validate="one_to_one")
metadata.to_csv(FINAL_METADATA_CSV, index=False)

print(metadata["status"].value_counts(dropna=False))
print("Metadata:", FINAL_METADATA_CSV)
metadata.head()


In [ ]:
# Final invariant check. Failed rows remain in metadata.csv with their error/status.
completed = metadata.loc[metadata["status"] == "complete", "final_path"]
invalid = [path for path in completed if not valid_final_png(Path(path))]
assert not invalid, f"Invalid final files: {invalid[:5]}"
print(f"Validated {len(completed):,} final 1024×1024 PNG images")
